In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# ===============================
# LOAD DATA
# ===============================
file_path = "production data with category.xlsx"
df = pd.read_excel(file_path)

# ===============================
# LABEL CREATION
# ===============================
df["Smart_Decision"] = (df["Production Gap"] <= 0).astype(int)

# ===============================
# CLEAN & FEATURE ENGINEERING
# ===============================

# Encode Category
df["Category_enc"] = df["Category"].map({
    "Runner": 0,
    "Repeater": 1,
    "Stranger": 2
})

# Encode Shift
df["Shift_enc"] = df["Shift"].map({"A": 0, "B": 1})

# Encode Machine
le_machine = LabelEncoder()
df["Machine_enc"] = le_machine.fit_transform(df["Machine ID"])

# Convert cycle times (example: "72/02" → 36)
def parse_cycle(x):
    if isinstance(x, str) and "/" in x:
        a, b = x.split("/")
        return float(a) / float(b)
    return np.nan

df["STD_Cycle"] = df["STD. Cycle Time"].apply(parse_cycle)
df["ACT_Cycle"] = df["Actual Standard Cycle Time"].apply(parse_cycle)

# ===============================
# SELECT FEATURES
# ===============================
features = [
    "Machine_enc",
    "Category_enc",
    "Shift_enc",
    "Required Qty",
    "Part Per Hour",
    "Hours Required",
    "Setup Rejections",
    "Actual Rejections",
    "STD_Cycle",
    "ACT_Cycle"
]

X = df[features].fillna(0)
y = df["Smart_Decision"]

# ===============================
# TRAIN / TEST SPLIT
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ===============================
# TRAIN MODEL
# ===============================
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

# ===============================
# EVALUATION
# ===============================
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
DATA_FILE = "production data with category.xlsx"
df = pd.read_excel(DATA_FILE)


In [ ]:
df["Smart_Decision"] = (df["Production Gap"] <= 0).astype(int)


In [ ]:
# Encode Category
df["Category_enc"] = df["Category"].map({
    "Runner": 0,
    "Repeater": 1,
    "Stranger": 2
})

# Encode Shift
df["Shift_enc"] = df["Shift"].map({"A": 0, "B": 1})

# Encode Machine
le_machine = LabelEncoder()
df["Machine_enc"] = le_machine.fit_transform(df["Machine ID"])

# Parse cycle time like "72/02"
def parse_cycle(x):
    if isinstance(x, str) and "/" in x:
        a, b = x.split("/")
        return float(a) / float(b)
    return np.nan

df["STD_Cycle"] = df["STD. Cycle Time"].apply(parse_cycle)
df["ACT_Cycle"] = df["Actual Standard Cycle Time"].apply(parse_cycle)


In [ ]:
FEATURES = [
    "Machine_enc",
    "Category_enc",
    "Shift_enc",
    "Required Qty",
    "Part Per Hour",
    "Hours Required",
    "Setup Rejections",
    "Actual Rejections",
    "STD_Cycle",
    "ACT_Cycle"
]

X = df[FEATURES].fillna(0)
y = df["Smart_Decision"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)


In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
parts_today = pd.DataFrame([
    {
        "Part": "P1",
        "Category": "Stranger",
        "Planned_Qty": 300,
        "Part_Per_Hour": 50,
        "STD_Cycle": 36,
        "ACT_Cycle": 34,
        "Eligible_Machines": ["MP-01", "MP-05"]
    },
    {
        "Part": "P2",
        "Category": "Repeater",
        "Planned_Qty": 250,
        "Part_Per_Hour": 80,
        "STD_Cycle": 28,
        "ACT_Cycle": 27,
        "Eligible_Machines": ["MP-01", "MP-10"]
    }
])


In [ ]:
candidates = []

for _, part in parts_today.iterrows():
    for m in part["Eligible_Machines"]:
        candidates.append({
            "Part": part["Part"],
            "Machine ID": m,
            "Category": part["Category"],
            "Shift": "A",
            "Required Qty": part["Planned_Qty"],
            "Part Per Hour": part["Part_Per_Hour"],
            "Hours Required": part["Planned_Qty"] / part["Part_Per_Hour"],
            "Setup Rejections": 0,
            "Actual Rejections": 0,
            "STD_Cycle": part["STD_Cycle"],
            "ACT_Cycle": part["ACT_Cycle"]
        })

candidates = pd.DataFrame(candidates)


In [ ]:
candidates["Category_enc"] = candidates["Category"].map({
    "Runner": 0,
    "Repeater": 1,
    "Stranger": 2
})

candidates["Shift_enc"] = candidates["Shift"].map({"A": 0, "B": 1})
candidates["Machine_enc"] = le_machine.transform(candidates["Machine ID"])

X_candidates = candidates[FEATURES].fillna(0)


In [ ]:
candidates["Smart_Score"] = model.predict_proba(X_candidates)[:, 1]
candidates = candidates.sort_values("Smart_Score", ascending=False)


In [ ]:
MAX_HOURS = 22
CHANGEOVER = 40 / 60
MAX_PARTS = 3

machines = candidates["Machine ID"].unique()

machine_hours = {m: 0 for m in machines}
machine_parts = {m: 0 for m in machines}

schedule = []

for _, row in candidates.iterrows():

    m = row["Machine ID"]
    hrs = row["Hours Required"]

    if machine_hours[m] + hrs + CHANGEOVER > MAX_HOURS:
        continue

    if machine_parts[m] >= MAX_PARTS:
        continue

    machine_hours[m] += hrs + CHANGEOVER
    machine_parts[m] += 1

    schedule.append({
        "Machine": m,
        "Part": row["Part"],
        "Qty": row["Required Qty"],
        "Smart_Score": round(row["Smart_Score"], 3)
    })


In [ ]:
final_plan = pd.DataFrame(schedule)
print("\nFINAL ML-GUIDED PRODUCTION PLAN\n")
print(final_plan)
